# 00 — First principles of compliance automation

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/04_compliance_auditor/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity (embeddings, similarity)
- Understanding of text processing

**What you will learn**:
- Why compliance is fundamentally a *mapping* problem between regulations and evidence
- Why keyword matching fails for regulatory language
- How regulatory requirements form nested, conditional trees
- That coverage is a spectrum, not a binary outcome
- Why continuous monitoring matters as regulations evolve
- The $35 B+ GRC market opportunity

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0" "scikit-learn>=1.2.0" "matplotlib>=3.7.0"

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from collections import defaultdict
import re, textwrap, json

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What is compliance?

At its core, compliance is a **mapping problem**: for every regulatory requirement **R**,
the organisation must demonstrate that evidence **E** exists.

```
Compliance = ∀ R ∈ Regulations : ∃ E ∈ Evidence such that E satisfies R
```

Consider **GDPR Article 17** — the *right to erasure*:

> "The data subject shall have the right to obtain from the controller the erasure of
> personal data concerning him or her without undue delay…"

The company must map this to concrete artefacts — a deletion policy, a technical
procedure, user-facing documentation, and audit logs proving execution.

In [ ]:
# A requirement-to-evidence mapping for a handful of GDPR articles
@dataclass
class Requirement:
    """Single regulatory requirement with obligation metadata."""
    id: str
    article: str
    text: str
    obligation: str  # must | should | may

@dataclass
class Evidence:
    """A piece of corporate evidence that may satisfy requirements."""
    id: str
    source: str
    text: str

# ── Sample requirements ──
requirements: List[Requirement] = [
    Requirement("GDPR-17-1", "Art. 17(1)", "The data subject shall have the right to obtain erasure of personal data without undue delay.", "must"),
    Requirement("GDPR-15-1", "Art. 15(1)", "The data subject shall have the right to obtain confirmation of processing and access to personal data.", "must"),
    Requirement("GDPR-20-1", "Art. 20(1)", "The data subject shall have the right to receive personal data in a structured, machine-readable format.", "must"),
    Requirement("GDPR-25-1", "Art. 25(1)", "The controller shall implement appropriate technical and organisational measures for data protection by design.", "must"),
    Requirement("GDPR-32-1", "Art. 32(1)", "The controller shall implement appropriate technical measures to ensure security of processing.", "must"),
]

# ── Sample company evidence ──
evidences: List[Evidence] = [
    Evidence("POL-01", "Privacy Policy §4", "Users can request complete deletion of their account and associated data at any time."),
    Evidence("POL-02", "Privacy Policy §3", "Users may request a copy of all personal data we hold, delivered within 30 days."),
    Evidence("POL-03", "Data Export Guide", "Personal data can be exported in CSV and JSON format through the account settings page."),
    Evidence("POL-04", "Security Policy §2", "All data is encrypted at rest using AES-256 and in transit using TLS 1.3."),
    Evidence("POL-05", "Engineering SOP", "Privacy impact assessments are conducted before launching any new data-processing feature."),
]

# ── Build the mapping ──
mapping: Dict[str, List[str]] = {
    "GDPR-17-1": ["POL-01"],           # right to erasure → deletion policy
    "GDPR-15-1": ["POL-02"],           # access right → data copy process
    "GDPR-20-1": ["POL-03"],           # portability → export guide
    "GDPR-25-1": ["POL-05"],           # privacy by design → PIA process
    "GDPR-32-1": ["POL-04"],           # security → encryption policy
}

print("Requirement → Evidence mapping")
print("=" * 60)
for req in requirements:
    ev_ids = mapping.get(req.id, [])
    ev_texts = [e.text[:60] + "…" for e in evidences if e.id in ev_ids]
    print(f"\n{req.id} ({req.article})")
    print(f"  Req : {req.text[:80]}…")
    for et in ev_texts:
        print(f"  Evid: {et}")
print(f"\n✓ {len(mapping)} requirements mapped to evidence")

## Section 2 — Why keyword matching fails

Regulation uses formal, legal language while corporate policies use everyday business
language. A keyword matcher cannot bridge this gap:

| Regulation phrase | Policy phrase | Keyword overlap |
|---|---|---|
| "right to erasure" | "users can request account deletion" | **0 words** |
| "data portability" | "export your information as CSV" | **0 words** |
| "lawful basis for processing" | "we only collect data you agree to share" | **0 words** |

We need **semantic** matching — embeddings that capture *meaning* rather than surface tokens.

In [ ]:
def keyword_overlap(text_a: str, text_b: str) -> float:
    """Jaccard similarity on lowercased word sets."""
    words_a = set(text_a.lower().split())
    words_b = set(text_b.lower().split())
    if not words_a or not words_b:
        return 0.0
    return len(words_a & words_b) / len(words_a | words_b)

# ── Test pairs: regulation text vs policy text ──
pairs: List[Tuple[str, str]] = [
    ("The data subject shall have the right to erasure of personal data",
     "Users can request complete deletion of their account and associated data"),
    ("Right to data portability in a structured machine-readable format",
     "Personal data can be exported in CSV and JSON format through account settings"),
    ("The controller shall ensure lawful basis for processing",
     "We only collect data you explicitly agree to share with us"),
    ("Data protection impact assessment shall be carried out",
     "Privacy reviews are conducted before launching new features"),
    ("Personal data shall be kept for no longer than necessary",
     "We automatically delete inactive accounts after 24 months"),
    ("The controller shall implement appropriate technical measures",
     "All systems use AES-256 encryption and regular security audits"),
    ("Data subjects shall be informed of the purposes of processing",
     "Our privacy notice explains why we collect each type of information"),
    ("Consent shall be freely given specific informed and unambiguous",
     "Users check a box confirming they agree after reading the notice"),
    ("Records of processing activities shall be maintained",
     "We keep an internal register of every data flow and its purpose"),
    ("Transfers to third countries require appropriate safeguards",
     "International vendors sign standard contractual clauses before receiving data"),
]

print(f"{'Pair':<5} {'Keyword overlap':>16}  Regulation (excerpt) → Policy (excerpt)")
print("-" * 95)
for i, (reg, pol) in enumerate(pairs, 1):
    score = keyword_overlap(reg, pol)
    print(f"  {i:<4} {score:>14.1%}   {reg[:35]:35s} → {pol[:35]}")
avg_kw = np.mean([keyword_overlap(r, p) for r, p in pairs])
print(f"\nAverage keyword overlap: {avg_kw:.1%} — far too low for reliable matching")

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

reg_texts = [r for r, _ in pairs]
pol_texts = [p for _, p in pairs]

reg_embs = model.encode(reg_texts)
pol_embs = model.encode(pol_texts)

# Diagonal = true-pair similarity; off-diagonal = noise
sim_matrix = cosine_similarity(reg_embs, pol_embs)

print(f"{'Pair':<5} {'Keyword':>9} {'Embedding':>10}  Match?")
print("-" * 50)
for i in range(len(pairs)):
    kw = keyword_overlap(pairs[i][0], pairs[i][1])
    emb = sim_matrix[i, i]
    match = "✓" if emb > 0.45 else "✗"
    print(f"  {i+1:<4} {kw:>8.1%} {emb:>9.3f}   {match}")

avg_emb = np.mean([sim_matrix[i, i] for i in range(len(pairs))])
print(f"\nAvg keyword overlap : {avg_kw:.1%}")
print(f"Avg embedding sim   : {avg_emb:.3f}")
print("\n→ Embeddings capture semantic matches that keywords miss entirely")

## Section 3 — Regulatory structure: nested conditional requirements

Regulations are **hierarchical**. GDPR Article 6 lists *six* lawful bases for
processing, each with sub-conditions:

```
Article 6 — Lawfulness of processing
├── (1) Processing is lawful only if at least one applies:
│   ├── (a) consent
│   ├── (b) contractual necessity
│   ├── (c) legal obligation
│   ├── (d) vital interests
│   ├── (e) public interest / official authority
│   └── (f) legitimate interests (with balancing test)
├── (2) Member States may adapt (c) and (e)
├── (3) Basis for (c) and (e) must be laid down in law
└── (4) Compatible-purpose test criteria
```

An auditor must parse this tree, not flatten it into a single sentence.

In [ ]:
@dataclass
class RequirementNode:
    """A node in a hierarchical requirement tree."""
    id: str
    text: str
    obligation: str          # must | should | may
    condition: Optional[str] = None
    children: List["RequirementNode"] = field(default_factory=list)

    def display(self, indent: int = 0) -> str:
        prefix = "  " * indent + ("├── " if indent else "")
        cond = f" [IF {self.condition}]" if self.condition else ""
        lines = [f"{prefix}{self.id} ({self.obligation}){cond}: {self.text[:70]}"]
        for child in self.children:
            lines.append(child.display(indent + 1))
        return "\n".join(lines)

# Build Article 6 tree
art6 = RequirementNode(
    id="GDPR-6",
    text="Processing shall be lawful only if and to the extent that at least one basis applies",
    obligation="must",
    children=[
        RequirementNode("GDPR-6-1a", "Data subject has given consent to processing for specific purposes", "must",
                         condition="consent obtained"),
        RequirementNode("GDPR-6-1b", "Processing is necessary for performance of a contract", "must",
                         condition="contract exists"),
        RequirementNode("GDPR-6-1c", "Processing is necessary for compliance with a legal obligation", "must",
                         condition="legal obligation applies"),
        RequirementNode("GDPR-6-1d", "Processing is necessary to protect vital interests of the data subject", "must",
                         condition="vital interests at stake"),
        RequirementNode("GDPR-6-1e", "Processing is necessary for task in public interest or official authority", "must",
                         condition="public interest task"),
        RequirementNode("GDPR-6-1f", "Processing is necessary for legitimate interests except where overridden by data subject rights", "must",
                         condition="legitimate interest after balancing test"),
    ]
)
art6_2 = RequirementNode("GDPR-6-2", "Member States may maintain or introduce more specific provisions for (c) and (e)", "may")
art6_3 = RequirementNode("GDPR-6-3", "Basis for (c) and (e) shall be laid down in Union or Member State law", "must")
art6_4 = RequirementNode("GDPR-6-4", "Compatible-purpose test: controller shall take into account link, context, nature, consequences, safeguards", "must")

root = RequirementNode("Article-6", "Lawfulness of processing", "must", children=[art6, art6_2, art6_3, art6_4])

print("Requirement tree — GDPR Article 6")
print("=" * 70)
print(root.display())
print(f"\nTotal nodes: {1 + len(art6.children) + 3}")

## Section 4 — The coverage spectrum

Compliance is **not binary**. A requirement can be addressed to varying degrees:

| Score | Level | Description |
|-------|-------|-------------|
| 5 | **Fully covered** | Policy directly addresses requirement with specifics |
| 4 | **Substantially covered** | Policy addresses most aspects, minor details missing |
| 3 | **Partially covered** | Some relevant language but significant gaps |
| 2 | **Weakly addressed** | Tangentially related policy exists but does not satisfy |
| 1 | **Not addressed** | No relevant policy language found |

In [ ]:
@dataclass
class CoverageAssessment:
    """Assessment of how well a policy covers a requirement."""
    requirement_id: str
    requirement_text: str
    policy_excerpt: str
    score: int  # 1-5
    explanation: str

# ── Examples at each coverage level ──
examples: List[CoverageAssessment] = [
    CoverageAssessment(
        "GDPR-17-1",
        "Right to erasure of personal data without undue delay",
        "Users can request deletion of their account and all associated data via Settings > Delete Account. "
        "Requests are processed within 72 hours. Backups are purged within 30 days.",
        5,
        "Fully covered — specific mechanism, timeline, and backup handling described."
    ),
    CoverageAssessment(
        "GDPR-15-1",
        "Right to obtain confirmation and access to personal data",
        "Users may request a copy of their data by contacting support@company.com. "
        "We aim to respond within 30 days.",
        4,
        "Substantially covered — process exists but format and scope not specified."
    ),
    CoverageAssessment(
        "GDPR-25-1",
        "Data protection by design and by default",
        "Our engineering team follows best practices for security during development.",
        3,
        "Partially covered — vague reference to 'best practices' without specifics."
    ),
    CoverageAssessment(
        "GDPR-20-1",
        "Right to data portability in structured machine-readable format",
        "We value transparency and give users control over their information.",
        2,
        "Weakly addressed — general statement about control, no portability mechanism."
    ),
    CoverageAssessment(
        "GDPR-33-1",
        "Notification of personal data breach to supervisory authority within 72 hours",
        "",
        1,
        "Not addressed — no breach notification policy found."
    ),
]

print("Coverage spectrum examples")
print("=" * 80)
labels = {5: "Fully covered", 4: "Substantially", 3: "Partially", 2: "Weakly", 1: "Not addressed"}
for ex in examples:
    print(f"\n[{ex.score}/5] {labels[ex.score]:15s} | {ex.requirement_id}")
    print(f"  Requirement : {ex.requirement_text[:70]}")
    if ex.policy_excerpt:
        print(f"  Policy      : {ex.policy_excerpt[:70]}…")
    else:
        print(f"  Policy      : — none —")
    print(f"  Explanation : {ex.explanation}")

# Visualise
scores = [ex.score for ex in examples]
colors = ["#d32f2f", "#f57c00", "#fbc02d", "#7cb342", "#388e3c"]
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh([ex.requirement_id for ex in examples], scores, color=[colors[s-1] for s in scores])
ax.set_xlabel("Coverage score")
ax.set_title("Coverage spectrum across five example requirements")
ax.set_xlim(0, 5.5)
for i, s in enumerate(scores):
    ax.text(s + 0.1, i, f"{s}/5", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## Section 5 — Continuous monitoring

Regulations are **living documents**. The GDPR alone has had major enforcement
milestones, new SCCs, and Schrems II implications. A compliance system must:

1. **Detect changes** — diff old vs. new regulation text
2. **Classify impact** — new requirement, modified threshold, or editorial change
3. **Re-assess** — only re-audit the affected requirements

In [ ]:
import difflib

# ── Simulated regulation change ──
old_text = """Article 33 — Notification of a personal data breach
(1) In the case of a personal data breach, the controller shall without
undue delay and, where feasible, not later than 72 hours after becoming
aware of it, notify the personal data breach to the supervisory authority.
(2) The notification shall at least:
(a) describe the nature of the breach;
(b) communicate the DPO contact details;
(c) describe the likely consequences;
(d) describe the measures taken or proposed to address the breach."""

new_text = """Article 33 — Notification of a personal data breach
(1) In the case of a personal data breach, the controller shall without
undue delay and, where feasible, not later than 48 hours after becoming
aware of it, notify the personal data breach to the supervisory authority.
The notification must include an estimated number of affected data subjects.
(2) The notification shall at least:
(a) describe the nature and categories of the breach;
(b) communicate the DPO contact details;
(c) describe the likely consequences and severity assessment;
(d) describe the measures taken or proposed to address the breach;
(e) provide a timeline of events from detection to notification."""

diff = list(difflib.unified_diff(
    old_text.splitlines(), new_text.splitlines(),
    fromfile="Art33_v1", tofile="Art33_v2", lineterm=""
))

print("Regulation change diff — Article 33")
print("=" * 60)
for line in diff:
    if line.startswith("+") and not line.startswith("+++"):
        print(f"  \033[92m{line}\033[0m")  # green
    elif line.startswith("-") and not line.startswith("---"):
        print(f"  \033[91m{line}\033[0m")  # red
    else:
        print(f"  {line}")

# ── Classify changes ──
changes = [
    {"type": "threshold_change", "detail": "72 hours → 48 hours", "impact": "HIGH"},
    {"type": "new_requirement", "detail": "Estimated number of affected data subjects", "impact": "MEDIUM"},
    {"type": "modified_clause", "detail": "(a) now includes 'categories of the breach'", "impact": "MEDIUM"},
    {"type": "modified_clause", "detail": "(c) now requires 'severity assessment'", "impact": "MEDIUM"},
    {"type": "new_clause", "detail": "(e) timeline of events from detection to notification", "impact": "HIGH"},
]

print("\nClassified changes:")
for c in changes:
    print(f"  [{c['impact']:6s}] {c['type']:20s} — {c['detail']}")

In [ ]:
# ── GDPR enforcement timeline ──
timeline = [
    ("2016-04", "GDPR adopted by EU Parliament"),
    ("2018-05", "GDPR enforcement begins"),
    ("2019-01", "Google fined €50 M by CNIL (France)"),
    ("2021-07", "Amazon fined €746 M by Luxembourg DPA"),
    ("2021-09", "WhatsApp fined €225 M by Irish DPC"),
    ("2023-01", "Meta fined €390 M for behavioral advertising"),
    ("2023-05", "Meta fined €1.2 B for US data transfers"),
    ("2024-01", "Cumulative GDPR fines exceed €4.5 B"),
]

fig, ax = plt.subplots(figsize=(10, 3))
dates = list(range(len(timeline)))
for i, (date, event) in enumerate(timeline):
    color = "#d32f2f" if "fined" in event.lower() else "#1976d2"
    ax.scatter(i, 0, s=120, c=color, zorder=5)
    ax.annotate(f"{date}\n{event[:40]}", (i, 0),
                textcoords="offset points", xytext=(0, 15 if i % 2 == 0 else -30),
                ha="center", fontsize=7, rotation=0)
ax.set_xlim(-0.5, len(timeline) - 0.5)
ax.set_ylim(-1, 1)
ax.axhline(y=0, color="gray", linewidth=0.8)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title("GDPR enforcement timeline — key milestones")
plt.tight_layout()
plt.show()

## Section 6 — Market analysis: why this matters

The **Governance, Risk & Compliance (GRC)** market is valued at over **$35 billion** and
growing at ~14 % CAGR. Key data points:

| Metric | Value |
|--------|-------|
| GRC market size (2024) | $35 B+ |
| GDPR fines to date | €4.5 B+ |
| Typical manual audit cost | $50 K – $500 K |
| Audit cycle | 3 – 12 months |
| Vanta ARR | >$100 M |
| Drata ARR | >$100 M |

**Vanta** and **Drata** reached $100 M+ ARR by automating evidence *collection* — but
the semantic mapping between regulations and policies remains largely manual.
That is where LLM-powered compliance auditors create a step-function improvement.

In [ ]:
# ── Simple ROI model ──
manual_audit_cost = 150_000   # USD, mid-range
audits_per_year = 2
manual_annual = manual_audit_cost * audits_per_year

ai_platform_cost = 36_000    # annual subscription
ai_labor_reduction = 0.70    # 70 % labor savings
ai_annual = ai_platform_cost + manual_annual * (1 - ai_labor_reduction)

savings = manual_annual - ai_annual
roi = savings / ai_annual * 100

print("Compliance automation ROI model")
print("=" * 45)
print(f"Manual audit cost × 2/yr : ${manual_annual:>10,}")
print(f"AI platform + residual   : ${ai_annual:>10,.0f}")
print(f"Annual savings           : ${savings:>10,.0f}")
print(f"ROI                      : {roi:>9.0f} %")
print()
print("Additional benefits:")
for b in ["Continuous monitoring vs. point-in-time snapshots",
          "Faster remediation through automated gap identification",
          "Reduced risk of regulatory fines (avg GDPR fine: €2 M+)",
          "Scalable across multiple frameworks (SOC 2, HIPAA, ISO 27001)"]:
    print(f"  • {b}")

## Exercises

1. **Extend the mapping** — Add three more GDPR articles (e.g. Articles 33, 35, 44)
   to the requirement-evidence mapping in Section 1. Create both the requirements and
   matching evidence items.

2. **Synonym expansion** — The keyword matcher in Section 2 fails because of
   vocabulary mismatch. Build a `synonym_keyword_overlap` function that expands a
   small synonym dictionary (e.g. `erasure → deletion, portability → export`) before
   computing Jaccard similarity. How much does it improve?

3. **Multi-framework mapping** — Extend the `Requirement` dataclass with a
   `framework: str` field and add five SOC 2 Type II requirements alongside the GDPR
   ones. Which evidence items satisfy requirements across *both* frameworks?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Compliance is a mapping problem: for each regulation R, find evidence E |
| 2 | Keyword matching fails — regulation and policy language differ semantically |
| 3 | Embeddings bridge the vocabulary gap with >0.5 cosine similarity on true pairs |
| 4 | Regulatory requirements are hierarchical with conditions and sub-obligations |
| 5 | Coverage is a 1-5 spectrum, not binary pass/fail |
| 6 | Regulations change — continuous monitoring and diffing is essential |
| 7 | GRC is a $35 B+ market; AI reduces audit costs by 60-80 % |

## What's Next

In **01 — Architecture**, we design the end-to-end compliance auditor pipeline:
requirement extraction, policy retrieval, coverage assessment, gap analysis, and
remediation generation.

---

## References

1. European Parliament, *Regulation (EU) 2016/679 — General Data Protection Regulation*, 2016.
2. GDPR Enforcement Tracker — <https://www.enforcementtracker.com/>
3. Vanta, *The State of Compliance 2024* — <https://www.vanta.com/>
4. Gartner, *Market Guide for IT Governance, Risk and Compliance Solutions*, 2024.